# Observed SE Australia Fire-Season Amplification

Computes the **observed** SE Australia fire-season warming amplification factor from ERA5 data
and compares it to the CMIP6 model estimate (0.935) from notebook 02.

The amplification factor is:
```
amplification = SE_AU_warming_trend / global_warming_trend
```

Where:
- **SE AU trend**: linear trend in ERA5 fire-season (Oct–Mar) mean daily-maximum temperature, 1961–2020
- **Global trend**: linear trend in FaIR global mean surface temperature, 1961–2020

This notebook uses only data already on disk — no new downloads required.

## Why this matters

CMIP6 models give SE AU amplification of 0.935 — SE Australia warmed slightly *less* than the global
mean in the model ensemble. Whether the observed amplification is higher or lower than this informs
whether our ERA5 PR estimates are conservative or generous. The ratio of observed to modelled
amplification provides a first-order observational constraint on the PR.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import linregress

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW  = Path('../../data/raw')
PROC = Path('../../data/processed')
FIGS = Path('../../outputs/figures')

LAT_S, LAT_N = -44, -28
LON_W, LON_E = 138, 154
FIRE_MONTHS  = [10, 11, 12, 1, 2, 3]
CLIM_START, CLIM_END = 1961, 1990
TREND_START, TREND_END = 1961, 2020

ERA5_PATH = RAW / 'era5' / 'era5_mx2t_daily_se_australia_1961_2020.nc'
print(f'ERA5 file: {ERA5_PATH.exists()}  ({ERA5_PATH.stat().st_size/1e6:.1f} MB)')

## 1. SE Australia fire-season warming trend from ERA5

Compute fire-season (Oct–Mar) mean of daily maximum temperature for SE Australia.
Take the linear trend over 1961–2020.

In [ ]:
ds = xr.open_dataset(ERA5_PATH)
da = ds['mx2t']

lat_dim = 'latitude' if 'latitude' in da.dims else 'lat'
lon_dim = 'longitude' if 'longitude' in da.dims else 'lon'

# Area-weighted spatial mean
weights = np.cos(np.deg2rad(da[lat_dim])).broadcast_like(da)
ts_daily = da.weighted(weights).mean(dim=[lat_dim, lon_dim]).squeeze().to_series()
ts_daily.index = pd.to_datetime(ts_daily.index)
if ts_daily.mean() > 100:
    ts_daily -= 273.15

# Fire-season mean of daily max (all fire-season days per season)
ts_fs = ts_daily[ts_daily.index.month.isin(FIRE_MONTHS)].copy()
ts_fs_shifted = ts_fs.copy()
ts_fs_shifted.index = ts_fs_shifted.index - pd.DateOffset(months=9)
fire_season_mean = ts_fs_shifted.resample('YE').mean()  # mean, not max
fire_season_mean.index = fire_season_mean.index.year
fire_season_mean = fire_season_mean.dropna()

# Restrict to trend period
fs_trend = fire_season_mean.loc[TREND_START:TREND_END]

slope_au, intercept_au, r_au, p_au, se_au = linregress(fs_trend.index, fs_trend.values)

print(f'ERA5 SE AU fire-season mean mx2t ({TREND_START}–{TREND_END}):')
print(f'  Trend:  {slope_au*10:.4f} °C/decade  ({slope_au:.5f} °C/yr)')
print(f'  R²:     {r_au**2:.3f}')
print(f'  p-val:  {p_au:.4f}')
print(f'  Total warming ({TREND_START}–{TREND_END}): {slope_au*(TREND_END-TREND_START):.3f} °C')

## 2. Global warming trend from FaIR

Load FaIR global mean surface temperature (p50 median of 841-member ensemble).
Compute linear trend over the same 1961–2020 period.

In [ ]:
ft = pd.read_parquet(PROC / 'fair_global_temperature.parquet')
ft = ft.set_index('year')['t_p50']

# Restrict to same trend period
ft_trend = ft.loc[TREND_START:TREND_END]

slope_gl, intercept_gl, r_gl, p_gl, se_gl = linregress(ft_trend.index, ft_trend.values)

print(f'FaIR global mean temperature ({TREND_START}–{TREND_END}):')
print(f'  Trend:  {slope_gl*10:.4f} °C/decade  ({slope_gl:.5f} °C/yr)')
print(f'  R²:     {r_gl**2:.3f}')
print(f'  Total warming ({TREND_START}–{TREND_END}): {slope_gl*(TREND_END-TREND_START):.3f} °C')

## 3. Observed amplification factor

In [ ]:
obs_amplification = slope_au / slope_gl
cmip6_amplification = 0.935  # ensemble median from notebook 02 (ACCESS-CM2, ACCESS-ESM1-5)
amp_ratio = obs_amplification / cmip6_amplification  # correction factor

print('Amplification factor (SE AU fire-season trend / FaIR global trend):')
print(f'  Observed (ERA5):     {obs_amplification:.3f}')
print(f'  CMIP6 models:        {cmip6_amplification:.3f}  (ACCESS-CM2 + ACCESS-ESM1-5, notebook 02)')
print(f'  Ratio (obs/CMIP6):   {amp_ratio:.3f}')
print()
if amp_ratio > 1:
    print(f'  ERA5 shows MORE amplification than models predict ({(amp_ratio-1)*100:.0f}% higher)')
    print(f'  → Models underestimate SE AU fire-season warming → ERA5 PR is conservative')
    print(f'  → Obs-corrected PR is HIGHER than ERA5 bootstrap → upper constraint')
else:
    print(f'  ERA5 shows LESS amplification than models predict ({(1-amp_ratio)*100:.0f}% lower)')
    print(f'  → Models overestimate SE AU fire-season warming relative to global mean')
    print(f'  → Obs-corrected PR is LOWER than ERA5 bootstrap → alternative lower-bound scenario')
    print(f'  → ERA5 bootstrap remains our primary estimate; obs-constrained provides comparison')

# Load CMIP6 amplification details
af_cmip6 = pd.read_csv(PROC / 'au_amplification_factor.csv')
print()
print('CMIP6 per-model amplification:')
print(af_cmip6.to_string(index=False))

# Save
out = pd.DataFrame([{
    'source': 'ERA5_observed',
    'trend_global_per_yr': slope_gl,
    'trend_au_per_yr': slope_au,
    'amplification': obs_amplification,
    'period': f'{TREND_START}-{TREND_END}',
    'metric': 'fire_season_mean_mx2t',
}])
out.to_csv(PROC / 'observed_amplification_factor.csv', index=False)
print(f'\nSaved to data/processed/observed_amplification_factor.csv')

## 4. Visualisation — SE AU vs global warming trajectories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

years = fs_trend.index

# ── Left: raw warming trajectories ──
ax = axes[0]

# Normalise both to 1961–1990 baseline for comparison
clim_au = fire_season_mean.loc[CLIM_START:CLIM_END].mean()
clim_gl = ft.loc[CLIM_START:CLIM_END].mean()
au_anom = fire_season_mean.loc[TREND_START:TREND_END] - clim_au
gl_anom = ft.loc[TREND_START:TREND_END] - clim_gl

ax.plot(au_anom.index, au_anom.values, color='#FF5722', alpha=0.7, linewidth=1, label='ERA5 SE AU fire season')
ax.plot(gl_anom.index, gl_anom.values, color='#2196F3', alpha=0.7, linewidth=1, label='FaIR global mean')

# Trend lines
fit_au = intercept_au + slope_au * years - clim_au
fit_gl = intercept_gl + slope_gl * years - clim_gl
ax.plot(years, fit_au, color='#FF5722', linewidth=2.5, label=f'SE AU trend ({slope_au*10:.3f}°C/dec)')
ax.plot(years, fit_gl, color='#2196F3', linewidth=2.5, label=f'Global trend ({slope_gl*10:.3f}°C/dec)')
ax.axhline(0, color='k', linewidth=0.7, alpha=0.4)
ax.set_xlabel('Year')
ax.set_ylabel('Temperature anomaly (°C, vs 1961–1990)')
ax.set_title('SE Australia fire-season vs global warming\n(1961–2020)', fontsize=11)
ax.legend(fontsize=8)

# ── Right: amplification comparison ──
ax2 = axes[1]
# Exclude ERA5_observed row — it is appended to the CSV by a later cell and would cause
# a shape mismatch on re-runs
af_models = af_cmip6[af_cmip6['model'] != 'ERA5_observed']
sources = ['ACCESS-CM2\n(CMIP6)', 'ACCESS-ESM1-5\n(CMIP6)', 'CMIP6\nensemble', 'ERA5\n(observed)']
amps    = list(af_models['amplification'].values) + [cmip6_amplification, obs_amplification]
colors  = ['#90A4AE', '#90A4AE', '#607D8B', '#FF5722']
bars = ax2.bar(sources, amps, color=colors, width=0.55, alpha=0.85)
ax2.axhline(1.0, color='k', linewidth=1, linestyle='--', alpha=0.5, label='Global mean = 1.0')
ax2.axhline(obs_amplification, color='#FF5722', linewidth=1.5, linestyle=':', alpha=0.7)
for bar, amp in zip(bars, amps):
    ax2.text(bar.get_x() + bar.get_width()/2, amp + 0.02, f'{amp:.3f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_ylabel('Amplification factor')
ax2.set_title('SE AU amplification: CMIP6 models vs observed\n(fire-season warming / global warming)', fontsize=11)
ax2.set_ylim(0, max(amps) * 1.25)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGS / 'au_observed_amplification.png', bbox_inches='tight')
plt.show()

## 5. Apply observed amplification as PR correction in liability

The observed amplification ratio (ERA5 / CMIP6) scales the ERA5 bootstrap PR to produce an
observationally-constrained estimate. Whether this is an upward or downward correction depends
on whether ERA5 shows more or less amplification than the CMIP6 models.

**First-order correction**:
```
PR_obs_corrected = PR_era5 × (obs_amplification / cmip6_amplification)
```

This is an approximation. The ERA5 bootstrap remains the primary PR estimate. The corrected
values are reported here as an obs-constrained sensitivity check.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

PROC = Path('../../data/processed')

# ERA5 bootstrap PR values (from notebook 04)
boot = pd.read_parquet(PROC / 'black_summer_pr_era5_bootstrap.parquet')
pr_era5_p05 = float(boot['pr_boot'].quantile(0.05))
pr_era5_med = float(boot['pr_boot'].median())
pr_era5_p95 = float(boot['pr_boot'].quantile(0.95))

# Apply amplification correction
pr_obs_p05 = pr_era5_p05 * amp_ratio
pr_obs_med = pr_era5_med * amp_ratio
pr_obs_p95 = pr_era5_p95 * amp_ratio

def far(pr):
    return 1.0 - 1.0/pr if pr > 1 else 0.0

print('PR comparison:')
print(f'  ERA5 bootstrap:        p05={pr_era5_p05:.2f}  median={pr_era5_med:.2f}  p95={pr_era5_p95:.2f}')
print(f'  Obs-corrected (×{amp_ratio:.3f}): p05={pr_obs_p05:.2f}  median={pr_obs_med:.2f}  p95={pr_obs_p95:.2f}')
print()
print('FAR comparison:')
print(f'  ERA5 median:           {far(pr_era5_med):.3f}  ({far(pr_era5_med)*100:.1f}%)')
print(f'  Obs-corrected median:  {far(pr_obs_med):.3f}  ({far(pr_obs_med)*100:.1f}%)')
print(f'  WWA published:         ≥0.750 to ≥0.889 (validation reference)')

# Add corrected liability to parquet
lb = pd.read_parquet(PROC / 'black_summer_liability.parquet')
AUD_TO_USD = 0.69
D_CENTRAL  = 10.0 * AUD_TO_USD  # USD B

share = lb['cm_warming_share']
lb['liability_obs_p05_USD_M'] = share * far(pr_obs_p05) * D_CENTRAL * 1000
lb['liability_obs_med_USD_M'] = share * far(pr_obs_med) * D_CENTRAL * 1000
lb['liability_obs_p95_USD_M'] = share * far(pr_obs_p95) * D_CENTRAL * 1000

lb.to_parquet(PROC / 'black_summer_liability.parquet', index=False)

print()
print('Total CM liability — central damages (AUD 10B), USD B:')
print(f'  ERA5 conservative (p05):    {lb["liability_conservative_USD_M"].sum()/1000:.2f}')
print(f'  ERA5 central (median):      {lb["liability_central_USD_M"].sum()/1000:.2f}')
print(f'  Obs-corrected median:       {lb["liability_obs_med_USD_M"].sum()/1000:.2f}')
print(f'  Obs-corrected p95:          {lb["liability_obs_p95_USD_M"].sum()/1000:.2f}')
print()
print('Top 5 (ERA5 central vs obs-corrected median, USD M):')
lb_sorted = lb.sort_values('liability_central_USD_M', ascending=False)
print(lb_sorted.head(5)[['parent_entity','liability_central_USD_M','liability_obs_med_USD_M']].to_string(index=False))

## 6. Update amplification factor CSV with observed value

In [ ]:
af_cmip6 = pd.read_csv(PROC / 'au_amplification_factor.csv')
# Remove any prior ERA5_observed rows before appending the freshly computed one
af_models_only = af_cmip6[af_cmip6['model'] != 'ERA5_observed']
obs_row = pd.DataFrame([{
    'model': 'ERA5_observed',
    'trend_global': slope_gl,
    'trend_au': slope_au,
    'amplification': obs_amplification,
}])
af_all = pd.concat([af_models_only, obs_row], ignore_index=True)
af_all.to_csv(PROC / 'au_amplification_factor.csv', index=False)

print('Updated au_amplification_factor.csv:')
print(af_all.to_string(index=False))

## Key findings

**SE AU fire-season amplification (ERA5 observed vs CMIP6 models)**

| Source | Amplification | Metric | Period |
|--------|---------------|--------|--------|
| ACCESS-CM2 (CMIP6) | 1.030 | tasmax fire season / GMST | historical |
| ACCESS-ESM1-5 (CMIP6) | 0.841 | tasmax fire season / GMST | historical |
| CMIP6 ensemble median | 0.935 | tasmax fire season / GMST | historical |
| **ERA5 observed** | **0.726** | **mx2t fire season / FaIR GMST** | **1961–2020** |

- SE AU fire-season mean mx2t trend: **+0.14°C/decade** (ERA5, 1961–2020)
- FaIR global GMST trend: **+0.20°C/decade** (1961–2020)
- Observed amplification (0.726) is **lower** than CMIP6 models (0.935)
- Correction factor (obs/CMIP6) = **0.776** — models slightly overestimate relative fire-season warming

**Implication for liability estimates**: the obs-constrained PR is *lower* than the ERA5 bootstrap.
The ERA5 bootstrap median (PR=1.80) is the primary estimate; obs-corrected is an alternative lower-bound.

| Scenario | PR | FAR | Total CM liability (central damages) |
|----------|----|-----|--------------------------------------|
| ERA5 p05 | 1.00 | 0.0% | USD 0.00B |
| **ERA5 bootstrap median** | **1.80** | **44.4%** | **USD 3.07B** |
| ERA5 p95 | 2.86 | 65.0% | USD 4.48B |
| Obs-corrected median | 1.40 | 28.4% | USD 1.96B |
| Obs-corrected p95 | 2.22 | 54.9% | USD 3.79B |

**Interpretation note**: The metric here (fire-season mean mx2t) measures average warming,
not tail-probability change. The WWA PR (≥10) targets extreme heat days specifically.
The amplification-corrected estimate is a sensitivity check, not a replacement for the bootstrap PR.
The gap between ERA5 bootstrap (PR=1.8) and WWA (PR≥10) is substantial; it reflects the broader
natural-variability spread introduced by including all 4 hist-nat models (vs 2 previously).

→ See `wiki/findings/2026-05-24-observed-amplification.md`